# College Basketball ML - Exploratory Data Analysis

This notebook explores the collected game data and analyzes features for machine learning predictions.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import sys
sys.path.append('..')

from src.feature_engineering import CBBFeatureEngine

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Load Data

In [ ]:
# Connect to database
conn = sqlite3.connect('../data/database.db')

# Load games
games_df = pd.read_sql_query('SELECT * FROM games', conn)
games_df['date'] = pd.to_datetime(games_df['date'])

# Load team info
teams_df = pd.read_sql_query('SELECT * FROM team_info', conn)

conn.close()

print(f"Total games: {len(games_df):,}")
print(f"Total teams: {len(teams_df):,}")
print(f"\nSeasons: {sorted(games_df['season'].unique())}")
print(f"Date range: {games_df['date'].min()} to {games_df['date'].max()}")

In [ ]:
# Display sample games
games_df.head(10)

## 2. Basic Statistics

In [ ]:
# Calculate point differential
games_df['point_diff'] = games_df['home_score'] - games_df['away_score']
games_df['total_points'] = games_df['home_score'] + games_df['away_score']
games_df['home_win'] = (games_df['home_score'] > games_df['away_score']).astype(int)

# Summary statistics
print("\nGame Statistics:")
print(f"Average home score: {games_df['home_score'].mean():.1f}")
print(f"Average away score: {games_df['away_score'].mean():.1f}")
print(f"Average total points: {games_df['total_points'].mean():.1f}")
print(f"Average point differential: {abs(games_df['point_diff']).mean():.1f}")
print(f"\nHome win percentage: {games_df['home_win'].mean()*100:.1f}%")

## 3. Visualizations

In [ ]:
# Score distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Home vs Away Scores
axes[0, 0].scatter(games_df['home_score'], games_df['away_score'], alpha=0.5)
axes[0, 0].plot([40, 120], [40, 120], 'r--', label='Equal scores')
axes[0, 0].set_xlabel('Home Score')
axes[0, 0].set_ylabel('Away Score')
axes[0, 0].set_title('Home vs Away Scores')
axes[0, 0].legend()

# Point Differential Distribution
axes[0, 1].hist(games_df['point_diff'], bins=50, edgecolor='black')
axes[0, 1].axvline(x=0, color='r', linestyle='--', label='Tied')
axes[0, 1].set_xlabel('Point Differential (Home - Away)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Point Differential Distribution')
axes[0, 1].legend()

# Total Points Distribution
axes[1, 0].hist(games_df['total_points'], bins=30, edgecolor='black', color='green')
axes[1, 0].set_xlabel('Total Points')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Total Points Distribution')

# Games by Season
if 'season' in games_df.columns:
    season_counts = games_df.groupby('season').size()
    axes[1, 1].bar(season_counts.index, season_counts.values)
    axes[1, 1].set_xlabel('Season')
    axes[1, 1].set_ylabel('Number of Games')
    axes[1, 1].set_title('Games by Season')

plt.tight_layout()
plt.show()

## 4. Team Analysis

In [ ]:
# Calculate team statistics using feature engine
engine = CBBFeatureEngine()
team_stats = engine.calculate_basic_stats(games_df)

print(f"Calculated stats for {len(team_stats)} teams\n")
print("Top 10 teams by point differential:")
print(team_stats.nlargest(10, 'point_diff')[['team_name', 'wins', 'losses', 'ppg', 'opp_ppg', 'point_diff']])

In [ ]:
# Team performance visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# PPG vs Opp PPG
axes[0].scatter(team_stats['ppg'], team_stats['opp_ppg'], 
                s=team_stats['games']*20, alpha=0.6)
axes[0].plot([50, 90], [50, 90], 'r--', label='Equal')
axes[0].set_xlabel('Points Per Game')
axes[0].set_ylabel('Opponent Points Per Game')
axes[0].set_title('Team Offensive vs Defensive Performance')
axes[0].legend()

# Win % vs Point Differential
axes[1].scatter(team_stats['point_diff'], team_stats['win_pct'], alpha=0.6)
axes[1].set_xlabel('Average Point Differential')
axes[1].set_ylabel('Win Percentage')
axes[1].set_title('Point Differential vs Win Percentage')

plt.tight_layout()
plt.show()

## 5. Feature Engineering & Correlation

In [ ]:
# Engineer features
results = engine.engineer_all_features(rolling_windows=[5, 10])

if results and 'features' in results:
    # Get features for window size 5
    if 'window_5' in results['features']:
        features_df = results['features']['window_5']
        
        if len(features_df) > 0:
            print(f"\nFeatures created: {len(features_df)} games\n")
            
            # Select numeric columns for correlation
            numeric_cols = features_df.select_dtypes(include=[np.number]).columns
            exclude = ['game_id', 'season', 'home_team_id', 'away_team_id']
            numeric_cols = [col for col in numeric_cols if col not in exclude]
            
            # Correlation with target
            if 'point_differential' in numeric_cols:
                correlations = features_df[numeric_cols].corr()['point_differential'].sort_values(ascending=False)
                print("\nTop correlations with point differential:")
                print(correlations.head(10))
                
                # Correlation heatmap
                plt.figure(figsize=(12, 10))
                key_features = ['home_ppg_5', 'away_ppg_5', 'home_opp_ppg_5', 'away_opp_ppg_5',
                               'ppg_diff_5', 'def_diff_5', 'point_differential']
                key_features = [f for f in key_features if f in features_df.columns]
                
                if len(key_features) > 0:
                    sns.heatmap(features_df[key_features].corr(), annot=True, cmap='coolwarm', center=0)
                    plt.title('Feature Correlation Matrix')
                    plt.tight_layout()
                    plt.show()
        else:
            print("\nNot enough games to create matchup features yet.")
            print("Need multiple games per team to calculate rolling statistics.")
else:
    print("\nNo features generated - need more historical data.")

## 6. Next Steps

Once we have enough historical data:
1. Train baseline models
2. Perform hyperparameter tuning
3. Evaluate model performance
4. Create prediction pipeline

In [ ]:
# Check scraper progress
print("\nCurrent data status:")
print(f"Games collected: {len(games_df):,}")
print(f"Unique teams: {len(set(games_df['home_team_id'].unique()).union(set(games_df['away_team_id'].unique())))}")
print(f"\nContinue collecting historical data for better model training!")